# T2 - Transmit Beamforming (standalone)
Input: probe description + focus and apodization. Output: per-element transmit delays, apodization weights and the element pulses. The team's code is used unchanged.

In [ ]:
# --- Step 0: find the project root (works from any folder: VS Code, Jupyter, Colab) and add every module to sys.path
import sys, pathlib
_here = pathlib.Path.cwd().resolve()
ROOT = next((p for p in [_here, *_here.parents] if (p / "integration" / "03_full_pipeline" / "_paths.py").exists()), None)
if ROOT is None:
    raise SystemExit("Project root not found. Open the NITK-UsoundSim-v2 folder in VS Code (File > Open Folder) and run again.")
sys.path.insert(0, str(ROOT / "integration" / "03_full_pipeline"))
import _paths                                   # noqa: F401  (adds modules/01..05 and integration/01..03 to sys.path)
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline") if "ipykernel" in sys.modules else None
import matplotlib.pyplot as plt
import numpy as np
print("project root:", ROOT)

In [ ]:
T2 = ROOT / 'teams' / 'T2_TX_Beamforming' / 'Usound'
sys.path.insert(0, str(T2))
import json
from src.transducer_interface import TransducerConfig
from src.beamformer import build_tx_package, generate_element_pulses
from nitk_usoundsim import config

## 1. The probe
The pipeline's probe (128 elements, 5 MHz, lambda/2 pitch). The team's own JSON describes a Philips L12-4 model (8 MHz, 0.30 mm); load it by setting `USE_TEAM_JSON = True`.

In [ ]:
USE_TEAM_JSON = False
if USE_TEAM_JSON:
    tc = TransducerConfig.from_json(T2 / 'config' / 'transducer_interface.json')
else:
    tc = TransducerConfig('v2 pipeline probe', config.N_ELEMENTS, config.PITCH, 4e6, 12e6, config.F0)
print(tc)

## 2. Focused transmit (change the focus below)

In [ ]:
beam = json.loads((T2 / 'config' / 'beamforming.json').read_text())
beam.update(focus_x_m=0.0, focus_z_m=0.020, steering_angle_deg=0.0, apodization='hann')
pkg = build_tx_package(tc, beam)
x_mm = pkg['element_x_m']*1e3; d_us = pkg['tx_delays_s']*1e6
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(x_mm, d_us, marker='o', ms=3); ax[0].set(title='transmit delay per element', xlabel='element x (mm)', ylabel='delay (us)')
ax[1].plot(x_mm, pkg['tx_weights']); ax[1].set(title='apodization weights (hann)', xlabel='element x (mm)')
plt.tight_layout(); plt.show()
print(f"delay range {d_us.min():.3f} .. {d_us.max():.3f} us | f0 {pkg['tx_frequency_hz']/1e6:g} MHz")

## 3. Element pulses (what T3 would transmit)

In [ ]:
ts, ps = generate_element_pulses(pkg['tx_delays_s'], pkg['tx_frequency_hz'], pkg['tx_weights'])
plt.figure(figsize=(9, 3.5)); plt.imshow(ps, aspect='auto', cmap='seismic', extent=[ts[0]*1e6, ts[-1]*1e6, ps.shape[0], 0]); plt.xlabel('time (us)'); plt.ylabel('element'); plt.title(f'pulses {ps.shape}'); plt.show()

## 4. Compare focus depths

In [ ]:
for z in (10e-3, 20e-3, 30e-3):
    b = dict(beam, focus_z_m=z); p = build_tx_package(tc, b); plt.plot(p['element_x_m']*1e3, p['tx_delays_s']*1e6, label=f'focus {z*1e3:.0f} mm')
plt.xlabel('element x (mm)'); plt.ylabel('delay (us)'); plt.legend(); plt.show()